In [0]:
import requests
import json

# Get the authentication token from the notebook context
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
user_email = spark.sql("SELECT current_user() as user").collect()[0][0]

# Define the endpoint
base_url = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com"
endpoint = f"{base_url}/api/v1/ingest/fhir-bundle"

# Create a basic FHIR Bundle JSON
fhir_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "example-patient-001",
                "name": [
                    {
                        "use": "official",
                        "family": "Doe",
                        "given": ["John"]
                    }
                ],
                "gender": "male",
                "birthDate": "1980-01-01"
            },
            "request": {
                "method": "POST",
                "url": "Patient"
            }
        }
    ]
}

print("Testing FastAPI app accessibility...\n")
print(f"User: {user_email}")
print(f"Token: {token[:20]}...\n")

# Test 1: Root endpoint
print("1. Testing root endpoint (/)")
try:
    response = requests.get(base_url, headers={"Authorization": f"Bearer {token}"})
    print(f"   Status: {response.status_code}")
    print(f"   Response: {response.text[:200]}\n")
except Exception as e:
    print(f"   Error: {e}\n")

# Test 2: Docs endpoint (FastAPI automatic)
print("2. Testing /docs endpoint")
try:
    response = requests.get(f"{base_url}/docs", headers={"Authorization": f"Bearer {token}"})
    print(f"   Status: {response.status_code}")
    print(f"   Accessible: {response.status_code == 200}\n")
except Exception as e:
    print(f"   Error: {e}\n")

# Test 3: POST with Bearer token
print("3. POST with Authorization: Bearer <token>")
response = requests.post(
    endpoint,
    json=fhir_bundle,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
)
print(f"   Status: {response.status_code}")
print(f"   Response: {response.text}\n")

# Test 4: POST without authentication
print("4. POST without authentication")
response_no_auth = requests.post(
    endpoint,
    json=fhir_bundle,
    headers={"Content-Type": "application/json"}
)
print(f"   Status: {response_no_auth.status_code}")
print(f"   Response: {response_no_auth.text}\n")

print("="*60)
if response.status_code == 200:
    print("✓ SUCCESS! FHIR Bundle posted successfully")
    result = response.json()
    print(f"Bundle UUID: {result.get('bundle_uuid')}")
else:
    print("⚠ AUTHENTICATION ISSUE")
    print("\nThe FastAPI app needs authentication middleware to validate Bearer tokens.")
    print("The app is configured for UI access (Gradio), not direct API calls.")
    print("\nTo fix: Add authentication middleware to app.py that validates the Bearer token.")

## Solution: Add Authentication Middleware to app.py

Add this middleware code to your `app.py` file:

```python
from fastapi import HTTPException, Header
from typing import Optional
import requests

# Add this function to validate Databricks tokens
def verify_databricks_token(authorization: Optional[str] = Header(None)):
    """
    Validates Databricks workspace token by calling the Databricks API.
    """
    if not authorization:
        raise HTTPException(status_code=401, detail="Authorization header missing")
    
    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Invalid authorization format")
    
    token = authorization.replace("Bearer ", "")
    
    # Verify token by calling Databricks API
    try:
        response = requests.get(
            f"{WORKSPACE_URL}/api/2.0/preview/scim/v2/Me",
            headers={"Authorization": f"Bearer {token}"}
        )
        if response.status_code != 200:
            raise HTTPException(status_code=401, detail="Invalid or expired token")
        return response.json()  # Returns user info
    except Exception as e:
        raise HTTPException(status_code=401, detail=f"Token validation failed: {str(e)}")

# Update the endpoint to use the dependency
@app.post("/api/v1/ingest/fhir-bundle")
async def ingest_fhir_bundle(request: Request, user_info = Depends(verify_databricks_token)):
    # ... rest of your existing code
```

**Steps to implement:**
1. Open the app.py file at `/Users/matthew.giglia@databricks.com/synthea-on-fhir/zerobus/fhir_zerobus/src/zerobus_app/app.py`
2. Add the `verify_databricks_token` function after the imports
3. Update the endpoint decorator to include the `Depends(verify_databricks_token)` dependency
4. Redeploy the app

**Alternative (Quick Test)**: Make the endpoint public by removing authentication requirement (for testing only).

In [0]:
import requests
import json

# Get authentication token
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Endpoint
endpoint = "https://fhir-zerobus-ingest-himss2026-7474657999482942.aws.databricksapps.com/api/v1/ingest/fhir-bundle"

# FHIR Bundle
fhir_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "example-patient-001",
                "name": [{"use": "official", "family": "Doe", "given": ["John"]}],
                "gender": "male",
                "birthDate": "1980-01-01"
            },
            "request": {"method": "POST", "url": "Patient"}
        }
    ]
}

# POST request
response = requests.post(
    endpoint,
    json=fhir_bundle,
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    result = response.json()
    print(f"✓ Success! Bundle UUID: {result.get('bundle_uuid')}")
    print(json.dumps(result, indent=2))
else:
    print(f"✗ Failed: {response.text}")